In [2]:
import logging
import fastf1
import pandas as pd
import numpy as np

logging.getLogger('fastf1').setLevel(logging.CRITICAL)
fastf1.Cache.enable_cache('cache')

seasons = {2022}
races_per_season = {2022:2 }

all_laps=[]
all_weather = []
all_trackStatus = []
all_telemetry = []


In [ ]:
for season in seasons:
    for round_num in range(1, races_per_season[season]+1):
        try:
            session= fastf1.get_session(season,round_num,"R")
            session.load(laps=True,telemetry=True, weather=True)
            
            laps = session.laps.copy()
            laps['Season'] = season
            laps['Round'] = round_num
            laps = laps[['TrackStatus','Time','LapStartTime', 'Driver', 'LapNumber', 'LapTime', 
             'TyreLife', 'Sector1Time', 'Sector2Time', 'Sector3Time',
             'Stint', 'Compound', 'Position', 'TrackStatus', 'Season','Round']]
            laps=laps.sort_values(['LapStartTime'])
            all_laps.append(laps)
            

            weather = session.weather_data.copy()
            weather['Season'] = season
            weather['Round'] = round_num
            if 'Date' not in weather.columns:

                 weather['Date'] = session.date + weather['Time']


            weather = weather.sort_values('Time')
            all_weather.append(weather)
            

            # we loop here through the laps we got to get telemetry data per lap    
            # we add driver and lapnumber from the laps attributes to
            
            for _, lap in session.laps.iterrows():
                telemetry= lap.get_telemetry(frequency=1).copy()
                telemetry = telemetry[['Date', 'SessionTime', 'X', 'Y', 
                    'Z', 'Status', 'Speed', 'RPM', 'Throttle','Distance','RelativeDistance','DriverAhead','DistanceToDriverAhead','Time']]
                
                telemetry.loc[:, 'Season'] = season
                telemetry.loc[:, 'Round'] = round_num
                telemetry.loc[:, 'LapNumber'] = lap['LapNumber']
                telemetry.loc[:, 'Driver'] = lap['Driver']
               

                telemetry = telemetry.sort_values(['Date', 'SessionTime'])
                all_telemetry.append(telemetry)


        except Exception as e:
            print(f"Skipping Season {season} Round {round_num} due to the following error: {e}")


In [6]:
telemetry_df =pd.concat(all_telemetry)
laps_df=pd.concat(all_laps)
weather_df=pd.concat(all_weather)

telemetry_df = telemetry_df.sort_values(['Date'])
weather_df = weather_df.sort_values(['Date'])


In [7]:
telemetry_with_weather_df = pd.merge_asof(
    telemetry_df,
    weather_df,
    on='Date',
    direction='backward',
    suffixes=('', '_weather'),
    by=['Season', 'Round'] 
)

In [73]:


telemetry_with_weather_df.to_csv("telemetry_with_weather.csv",index=False)
telemetry_df.to_csv("telemetry.csv",index=False)
laps_df.to_csv("laps.csv",index=False)
weather_df.to_csv("weather.csv",index=False)


In [ ]:
# ---------------------------------------------------------
# DIRECT MERGE ON LAST ROW (Anti-Leakage)
# ---------------------------------------------------------

# 1. Sort the telemetry data chronologically.
# This ensures that the row with the maximum SessionTime is truly the last one.
telemetry_with_weather_df = telemetry_with_weather_df.sort_values(
    ['Season', 'Round', 'Driver', 'SessionTime']
)

# 2. Identify and isolate the LAST telemetry row for every lap.
# We group by the Lap ID and use .tail(1) to select only the final row of each group.
# This creates a small DataFrame containing only the desired merge targets.
last_telemetry_rows = telemetry_with_weather_df.groupby(
    ['Season', 'Round', 'Driver', 'LapNumber']
).tail(1).copy()


# 3. Merge the Lap Summaries ONLY onto the last rows DataFrame.
# This attaches the LapTime/SectorTime data to the final packet of the lap.
lap_summary_cols = laps_df.columns.difference(last_telemetry_rows.columns)
merge_keys = ['Season', 'Round', 'Driver', 'LapNumber']

merged_last_rows = pd.merge(
    last_telemetry_rows,
    laps_df[merge_keys.append(lap_summary_cols.tolist())],
    on=merge_keys,
    how='left',
    suffixes=('', '_lap') 
)


# 4. Final step: Combine the "last rows" back with the rest of the telemetry data.
# We use pd.concat to stitch the two parts back together.
# We exclude the indices that were selected for the last row merge.
body_telemetry_rows = telemetry_with_weather_df.drop(last_telemetry_rows.index)

# Note: The merge in Step 3 automatically handles the column creation (e.g., LapTime, Time_lap).
final_df = pd.concat([body_telemetry_rows, merged_last_rows]).sort_values(
    ['Season', 'Round', 'Driver', 'SessionTime']
)

# The result is telemetry data with NaN for lap summary, except for the final row.

KeyError: 'Season'